# Matched Filter for Gravitational Wave Detection — PyCBC

This notebook implements the same **matched filter** analysis as `matched_filter.ipynb` (ripple), but uses [PyCBC](https://pycbc.org/) — the production gravitational-wave analysis library.

It operates on data produced by **Step 1 (data_generation)** of the SparseBank pipeline. Each HDF5 file contains pre-whitened strain from two detectors (H1, L1) along with the ground-truth source parameters.

## Data format

| Dataset | Shape | Description |
|---------|-------|-------------|
| `injected_data` | `(N, 2, 28160)` | Whitened H1+L1 strain, 55 s at 512 Hz |
| `mass_1`, `mass_2` | `(N,)` | Component masses [$M_\odot$] |
| `chirp_mass`, `mass_ratio` | `(N,)` | Derived mass parameters |
| `chi1`, `chi2` | `(N,)` | Aligned spins |
| `distance` | `(N,)` | Luminosity distance [Mpc] |
| `snr` | `(N,)` | Network SNR |

## Signal timing

The data generation pipeline injects the signal such that the merger occurs ~64 s from the start of the window. Since only the first 55 s are stored, the data contains the **pre-merger inspiral chirp**. The matched filter SNR peak will appear near the end of the analysis window.

## Outline

1. Load HDF5 data and inspect the file structure
2. Visualise the whitened strain
3. Nominal PSD for template whitening
4. Generate template — `get_fd_waveform` with TaylorF2
5. Whiten template and run `pycbc.filter.matched_filter` with flat PSD
6. Template bank search over a $(m_1, m_2)$ grid
7. Match and fitting factor — `pycbc.filter.match`
8. Power $\chi^2$ veto
9. Re-weighted SNR
10. Comparison with ripple results

## 1. Imports

In [ ]:
%config InlineBackend.figure_format = 'retina'

import h5py
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.signal import spectrogram

import pycbc
import pycbc.psd
import pycbc.waveform
import pycbc.filter
import pycbc.vetoes
from pycbc.types import FrequencySeries, TimeSeries

print(f"PyCBC version : {pycbc.__version__}")

## 2. Configuration

Set `HDF5_PATH` to a file produced by Step 1 of the SparseBank pipeline.

In [ ]:
# ── Path to a Step-1 HDF5 file ────────────────────────────────────────────────
HDF5_PATH = Path("path/to/data/test/sig_combined_test_0.h5")   # <── set this

# ── Pipeline parameters (must match data_generation config) ──────────────────
SAMPLE_RATE = 512    # Hz
F_LOW       = 20.0   # Hz
F_HIGH      = 256.0  # Hz  (= SAMPLE_RATE / 2)
F_REF       = 50.0   # Hz
DATA_DUR    = 55.0   # s  — stored window length

# ── Analysis parameters ───────────────────────────────────────────────────────
# Zero-pad to T_PAD so the matched filter reaches the coalescence time (~64 s).
T_PAD   = 128.0
DELTA_T = 1.0 / SAMPLE_RATE
DELTA_F = 1.0 / T_PAD
N_PAD   = int(T_PAD * SAMPLE_RATE)
FLEN    = N_PAD // 2 + 1

EVENT_IDX = 0   # which event to analyse inside the batch

## 3. Load HDF5 Data

In [ ]:
with h5py.File(HDF5_PATH, "r") as f:
    print("Datasets in file:")
    for key in f.keys():
        print(f"  {key:20s}  shape={f[key].shape}  dtype={f[key].dtype}")

    strain_all = f["injected_data"][:]          # (N, 2, n_time)
    params = {k: f[k][:] for k in f.keys() if k != "injected_data"}

print(f"\nBatch size    : {strain_all.shape[0]}")
print(f"Channels      : {strain_all.shape[1]}  (H1, L1)")
print(f"Time samples  : {strain_all.shape[2]}  ({strain_all.shape[2]/SAMPLE_RATE:.1f} s at {SAMPLE_RATE} Hz)")

In [ ]:
# ── Extract one event ─────────────────────────────────────────────────────────
strain_H1_np = strain_all[EVENT_IDX, 0, :].astype(np.float64)
strain_L1_np = strain_all[EVENT_IDX, 1, :].astype(np.float64)

m1_true   = float(params["mass_1"][EVENT_IDX])
m2_true   = float(params["mass_2"][EVENT_IDX])
chi1_true = float(params["chi1"][EVENT_IDX])
chi2_true = float(params["chi2"][EVENT_IDX])
dist_true = float(params["distance"][EVENT_IDX])
snr_true  = float(params["snr"][EVENT_IDX])

print(f"Event index   : {EVENT_IDX}")
print(f"m1            : {m1_true:.3f} M_sun")
print(f"m2            : {m2_true:.3f} M_sun")
print(f"chi1, chi2    : {chi1_true:.3f}, {chi2_true:.3f}")
print(f"Distance      : {dist_true:.1f} Mpc")
print(f"Network SNR   : {snr_true:.2f}")

## 4. Visualise the Whitened Strain

In [ ]:
t_data = np.arange(len(strain_H1_np)) / SAMPLE_RATE

fig, axes = plt.subplots(2, 1, figsize=(12, 4), sharex=True)
for ax, strain, ifo in zip(axes, [strain_H1_np, strain_L1_np], ["H1", "L1"]):
    ax.plot(t_data, strain, lw=0.4, color='steelblue')
    ax.set_ylabel(f"{ifo} whitened strain")
    ax.grid(True, alpha=0.3)
axes[-1].set_xlabel("Time [s]")
fig.suptitle(
    f"Whitened strain — $m_1={m1_true:.2f}$, $m_2={m2_true:.2f}$ $M_\\odot$, "
    f"SNR$={snr_true:.1f}$",
    y=1.01,
)
plt.tight_layout()
plt.show()

In [ ]:
# Spectrogram of H1 to visualise the inspiral chirp
f_spec, t_spec, Sxx = spectrogram(
    strain_H1_np, fs=SAMPLE_RATE, nperseg=256, noverlap=248
)
fig, ax = plt.subplots(figsize=(12, 4))
ax.pcolormesh(t_spec, f_spec, np.log1p(Sxx), shading='gouraud', cmap='inferno')
ax.set_ylim(F_LOW, 200)
ax.set_xlabel("Time [s]")
ax.set_ylabel("Frequency [Hz]")
ax.set_title("H1 spectrogram (whitened) — PyCBC pipeline")
plt.tight_layout()
plt.show()

## 5. Nominal PSD and Template Whitening

The data was whitened during Step 1 using a PSD estimated from real LIGO background noise. We use PyCBC's built-in `aLIGOZeroDetHighPower` as the nominal PSD to whiten the template with the same spectral shape.

After whitening, the effective data PSD is approximately flat, so we run `matched_filter` with a **flat PSD** of ones.

In [ ]:
# aLIGO PSD — used only to whiten the template
psd_aligo = pycbc.psd.aLIGOZeroDetHighPower(FLEN, DELTA_F, F_LOW)
psd_aligo_arr = np.array(psd_aligo)

# Flat PSD for matched filtering (data is pre-whitened)
psd_flat_arr = np.ones(FLEN)
f_rfft = np.fft.rfftfreq(N_PAD, d=DELTA_T)
psd_flat_arr[f_rfft < F_LOW]  = np.inf
psd_flat_arr[f_rfft > F_HIGH] = np.inf
psd_flat_arr[0]               = np.inf
psd_flat = FrequencySeries(psd_flat_arr, delta_f=DELTA_F)

# Plot
f_plot_arr = np.array(psd_aligo.sample_frequencies)
valid = psd_aligo_arr > 0
fig, ax = plt.subplots(figsize=(8, 3))
ax.loglog(f_plot_arr[valid], np.sqrt(psd_aligo_arr[valid]), color='steelblue',
          label='aLIGO design PSD (template whitening)')
ax.axvspan(F_LOW, F_HIGH, alpha=0.1, color='green', label='Analysis band')
ax.set_xlabel('Frequency [Hz]')
ax.set_ylabel(r'$\sqrt{S_n(f)}$ [Hz$^{-1/2}$]')
ax.set_title('Nominal PSD — PyCBC built-in')
ax.legend()
ax.grid(True, which='both', alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Generate and Whiten Template

We use `pycbc.waveform.get_fd_waveform` with the `TaylorF2` approximant and the true parameters loaded from the HDF5 file, then whiten the template:

$$\tilde{h}_w(f) = \frac{\tilde{h}(f)}{\sqrt{S_n(f)}}$$

In [ ]:
hp_raw, _ = pycbc.waveform.get_fd_waveform(
    approximant = 'TaylorF2',
    mass1       = m1_true,
    mass2       = m2_true,
    spin1z      = chi1_true,
    spin2z      = chi2_true,
    distance    = 1.0,        # arbitrary — normalised by matched filter
    f_lower     = F_LOW,
    f_final     = F_HIGH,
    delta_f     = DELTA_F,
)
hp_raw.resize(FLEN)

print(f"Template length : {len(hp_raw)} bins")
print(f"Non-zero bins   : {(np.abs(np.array(hp_raw)) > 0).sum()}")

# Whiten: h_w(f) = h(f) / sqrt(S_n(f))
hp_arr        = np.array(hp_raw, dtype=complex)
safe_psd_arr  = np.where(psd_aligo_arr > 0, psd_aligo_arr, np.inf)
hp_w_arr      = hp_arr / np.sqrt(safe_psd_arr)

# Apply band mask
band_mask = (f_rfft >= F_LOW) & (f_rfft <= F_HIGH)
hp_w_arr  = np.where(band_mask, hp_w_arr, 0.0 + 0.0j)

hp_whitened = FrequencySeries(hp_w_arr, delta_f=DELTA_F)

print(f"Max |h_w(f)|    : {np.abs(hp_w_arr).max():.3e}")

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 5), sharex=True)

hp_amp = np.abs(np.array(hp_raw))
hp_w_amp = np.abs(hp_w_arr)

axes[0].semilogy(f_rfft[band_mask], hp_amp[band_mask],
                 color='steelblue', label='Raw template $|\\tilde{h}(f)|$')
axes[0].set_ylabel('Amplitude [Hz$^{-1}$]')
axes[0].legend()
axes[0].grid(True, which='both', alpha=0.3)
axes[0].set_title(
    f'TaylorF2 template — $m_1={m1_true:.2f}$, $m_2={m2_true:.2f}$ $M_\\odot$ — PyCBC'
)

axes[1].semilogy(f_rfft[band_mask], hp_w_amp[band_mask],
                 color='darkorange', label='Whitened template $|\\tilde{h}_w(f)|$')
axes[1].set_xlabel('Frequency [Hz]')
axes[1].set_ylabel('Whitened amplitude')
axes[1].legend()
axes[1].grid(True, which='both', alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Matched Filter

We zero-pad the 55-second whitened strain to 128 s, wrap it in a PyCBC `TimeSeries`, and call `pycbc.filter.matched_filter` with the whitened template and the flat PSD.

Because the data is pre-whitened, using a **flat PSD** is equivalent to the standard matched filter against coloured noise.

In [ ]:
# Zero-pad data to N_PAD and wrap in PyCBC TimeSeries
n_data = len(strain_H1_np)
strain_pad = np.zeros(N_PAD)
strain_pad[:n_data] = strain_H1_np

strain_ts = TimeSeries(strain_pad, delta_t=DELTA_T)
stilde    = strain_ts.to_frequencyseries()

# Ensure template and data have the same length
hp_whitened.resize(len(stilde))

# Matched filter with flat PSD
snr_ts = pycbc.filter.matched_filter(
    hp_whitened,
    stilde,
    psd                  = psd_flat,
    low_frequency_cutoff = F_LOW,
    high_frequency_cutoff= F_HIGH,
)

# Crop edge effects: remove ~4 s on each side
crop_s = 4.0
snr_cropped = snr_ts.crop(crop_s, crop_s)

snr_arr = np.abs(np.array(snr_cropped))
t_snr   = np.array(snr_cropped.sample_times)

peak_idx  = np.argmax(snr_arr)
peak_snr  = snr_arr[peak_idx]
peak_time = t_snr[peak_idx]

print(f"Peak SNR   : {peak_snr:.2f}")
print(f"Peak time  : {peak_time:.3f} s")
print(f"(merger expected ~64 s from data start; data stored 0–55 s)")

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 6))

# Full SNR
axes[0].plot(t_snr, snr_arr, lw=0.6, color='steelblue')
axes[0].axvline(DATA_DUR, color='grey', ls='--', lw=1.2,
                label=f'End of stored data ({DATA_DUR} s)')
axes[0].axvline(peak_time, color='tomato', ls=':', lw=1.5,
                label=f'SNR peak at {peak_time:.2f} s')
axes[0].set_ylabel('SNR')
axes[0].set_title('Matched Filter SNR — PyCBC (full time series, zero-padded)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Zoom around peak
zoom_win = 5.0
t_lo = max(t_snr[0], peak_time - zoom_win)
t_hi = min(t_snr[-1], peak_time + zoom_win)
mask_zoom = (t_snr >= t_lo) & (t_snr <= t_hi)
axes[1].plot(t_snr[mask_zoom], snr_arr[mask_zoom], lw=1.0, color='steelblue')
axes[1].axvline(DATA_DUR, color='grey', ls='--', lw=1.2)
axes[1].axvline(peak_time, color='tomato', ls=':', lw=1.5)
axes[1].set_xlabel('Time [s]')
axes[1].set_ylabel('SNR')
axes[1].set_title(f'Zoom around SNR peak (±{zoom_win} s)')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 8. Template Bank Search

We scan a grid of $(m_1, m_2)$ templates, running `matched_filter` for each and recording the peak SNR.

In [ ]:
m1_grid = np.linspace(1.0, 2.5, 10)
m2_grid = np.linspace(1.0, 2.3, 9)

bank_results = []

for m1_t in m1_grid:
    for m2_t in m2_grid:
        if m2_t > m1_t:
            continue

        hp_t, _ = pycbc.waveform.get_fd_waveform(
            approximant = 'TaylorF2',
            mass1       = m1_t,
            mass2       = m2_t,
            spin1z      = 0.0,
            spin2z      = 0.0,
            distance    = 1.0,
            f_lower     = F_LOW,
            f_final     = F_HIGH,
            delta_f     = DELTA_F,
        )
        hp_t.resize(FLEN)

        # Whiten
        hp_t_arr = np.array(hp_t, dtype=complex) / np.sqrt(safe_psd_arr)
        hp_t_arr = np.where(band_mask, hp_t_arr, 0.0 + 0.0j)
        hp_t_w   = FrequencySeries(hp_t_arr, delta_f=DELTA_F)
        hp_t_w.resize(len(stilde))

        snr_t = pycbc.filter.matched_filter(
            hp_t_w, stilde,
            psd                   = psd_flat,
            low_frequency_cutoff  = F_LOW,
            high_frequency_cutoff = F_HIGH,
        )
        try:
            snr_t_crop = snr_t.crop(crop_s, crop_s)
        except Exception:
            snr_t_crop = snr_t

        snr_t_arr = np.abs(np.array(snr_t_crop))
        t_t_arr   = np.array(snr_t_crop.sample_times)

        local_peak = snr_t_arr.max()
        local_time = float(t_t_arr[np.argmax(snr_t_arr)])
        bank_results.append((m1_t, m2_t, float(local_peak), local_time))

bank_results = np.array(bank_results)
best_idx = np.argmax(bank_results[:, 2])
best = bank_results[best_idx]

print(f"Best template  : m1 = {best[0]:.2f}, m2 = {best[1]:.2f} M_sun")
print(f"Best SNR       : {best[2]:.2f}")
print(f"True params    : m1 = {m1_true:.2f}, m2 = {m2_true:.2f} M_sun  (SNR={snr_true:.1f})")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
sc = ax.scatter(bank_results[:, 0], bank_results[:, 1],
                c=bank_results[:, 2], cmap='viridis', s=80,
                edgecolors='k', lw=0.5)
plt.colorbar(sc, ax=ax, label='Peak SNR')
ax.scatter(m1_true, m2_true, marker='*', s=300, color='tomato',
           zorder=5, label='True parameters')
ax.scatter(best[0], best[1], marker='D', s=120, color='lime',
           zorder=4, label='Best template')
ax.set_xlabel(r'$m_1$ [$M_\odot$]')
ax.set_ylabel(r'$m_2$ [$M_\odot$]')
ax.set_title('Template Bank SNR Map — PyCBC')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 9. Match and Fitting Factor

`pycbc.filter.match` computes the overlap between two waveforms, maximised over time and phase. We use whitened templates and a flat PSD to be consistent with the analysis above.

In [ ]:
matches = []
for row in bank_results:
    m1_t, m2_t = row[0], row[1]
    hp_t, _ = pycbc.waveform.get_fd_waveform(
        approximant='TaylorF2', mass1=m1_t, mass2=m2_t,
        spin1z=0.0, spin2z=0.0, distance=1.0,
        f_lower=F_LOW, f_final=F_HIGH, delta_f=DELTA_F,
    )

    tlen = max(len(hp_raw), len(hp_t))

    # Whiten both templates
    hp_sig_r = hp_raw.copy(); hp_sig_r.resize(tlen)
    hp_t.resize(tlen)

    psd_r_arr = np.array(pycbc.psd.aLIGOZeroDetHighPower(tlen//2+1, DELTA_F, F_LOW))
    safe_r    = np.where(psd_r_arr > 0, psd_r_arr, np.inf)

    hp_sig_w = FrequencySeries(np.array(hp_sig_r)/np.sqrt(safe_r), delta_f=DELTA_F)
    hp_t_w   = FrequencySeries(np.array(hp_t)/np.sqrt(safe_r),     delta_f=DELTA_F)

    psd_flat_r_arr = np.ones(tlen//2+1)
    f_r = np.fft.rfftfreq(tlen, d=DELTA_T)
    psd_flat_r_arr[f_r < F_LOW]  = np.inf
    psd_flat_r_arr[f_r > F_HIGH] = np.inf
    psd_flat_r = FrequencySeries(psd_flat_r_arr, delta_f=DELTA_F)

    m, _ = pycbc.filter.match(
        hp_sig_w, hp_t_w,
        psd=psd_flat_r, low_frequency_cutoff=F_LOW,
    )
    matches.append(m)

matches    = np.array(matches)
best_m_idx = matches.argmax()
print(f"Fitting factor : {matches.max():.4f}")
print(f"Best-match     : m1 = {bank_results[best_m_idx, 0]:.2f}, "
      f"m2 = {bank_results[best_m_idx, 1]:.2f} M_sun")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
sc = ax.scatter(bank_results[:, 0], bank_results[:, 1],
                c=matches, cmap='plasma', vmin=0.8, vmax=1.0,
                s=80, edgecolors='k', lw=0.5)
plt.colorbar(sc, ax=ax, label='Match')
ax.scatter(m1_true, m2_true, marker='*', s=300, color='cyan',
           zorder=5, label='True parameters')
ax.set_xlabel(r'$m_1$ [$M_\odot$]')
ax.set_ylabel(r'$m_2$ [$M_\odot$]')
ax.set_title('Template Match Map — PyCBC')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 10. Power $\chi^2$ Veto

We split the whitened template into equal-power sub-bands and verify that each contributes its expected fraction of the SNR. For a genuine signal, $\chi^2_r \approx 1$.

In [ ]:
num_bins = 16

chisq_ts = pycbc.vetoes.power_chisq(
    hp_whitened,
    stilde,
    num_bins,
    psd_flat,
    low_frequency_cutoff  = F_LOW,
    high_frequency_cutoff = F_HIGH,
)
chisq_r_ts = chisq_ts / (2 * num_bins - 2)

try:
    chisq_r_crop = chisq_r_ts.crop(crop_s, crop_s)
except Exception:
    chisq_r_crop = chisq_r_ts

chisq_arr = np.array(chisq_r_crop)
t_chisq   = np.array(chisq_r_crop.sample_times)

# Value at the SNR peak
tc_idx = np.argmin(np.abs(t_chisq - peak_time))
print(f"Reduced chi-sq at SNR peak : {chisq_arr[tc_idx]:.3f}  (expect ~1 for a signal)")

In [ ]:
# Align to common time axis
min_len = min(len(snr_arr), len(chisq_arr))
t_common     = t_snr[:min_len]
snr_common   = snr_arr[:min_len]
chisq_common = chisq_arr[:min_len]

zoom_win = 5.0
mask_plot = (t_common >= peak_time - zoom_win) & (t_common <= peak_time + zoom_win)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

ax1.plot(t_common[mask_plot], snr_common[mask_plot], color='steelblue', lw=1.0, label='SNR')
ax1.axvline(DATA_DUR, color='grey', ls='--', lw=1.2, label=f'End of data ({DATA_DUR} s)')
ax1.axvline(peak_time, color='tomato', ls=':', lw=1.5, label=f'Peak at {peak_time:.2f} s')
ax1.set_ylabel('SNR')
ax1.legend()
ax1.grid(True, alpha=0.3)
ax1.set_title(r'SNR and Power $\chi^2$ Veto — PyCBC (zoom around peak)')

ax2.plot(t_common[mask_plot], chisq_common[mask_plot], color='darkorange', lw=1.0, label=r'$\chi^2_r$')
ax2.axhline(1.0, color='grey', ls='--', alpha=0.6, label=r'$\chi^2_r = 1$')
ax2.axvline(DATA_DUR, color='grey', ls='--', lw=1.2)
ax2.axvline(peak_time, color='tomato', ls=':', lw=1.5)
ax2.set_xlabel('Time [s]')
ax2.set_ylabel(r'Reduced $\chi^2$')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 11. Re-weighted SNR

The PyCBC detection statistic penalises candidates with $\chi^2_r > 1$:

$$\hat{\rho} = \begin{cases} \rho & \chi^2_r \leq 1 \\ \rho \left[\tfrac{1}{2}(1 + (\chi^2_r)^3)\right]^{-1/6} & \chi^2_r > 1 \end{cases}$$

In [ ]:
def reweighted_snr(snr, chisq_r):
    rho_new = snr.copy()
    mask = chisq_r > 1.0
    rho_new[mask] = snr[mask] * (0.5 * (1.0 + chisq_r[mask]**3))**(-1.0/6)
    return rho_new


newsnr = reweighted_snr(snr_common, chisq_common)

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(t_common[mask_plot], snr_common[mask_plot],
        color='steelblue', lw=1.0, alpha=0.8, label=r'SNR $\rho$')
ax.plot(t_common[mask_plot], newsnr[mask_plot],
        color='darkorange', lw=1.5, ls='--', label=r'Re-weighted SNR $\hat{\rho}$')
ax.axvline(DATA_DUR, color='grey', ls='--', lw=1.2)
ax.axvline(peak_time, color='tomato', ls=':', lw=1.5, label=f'Peak at {peak_time:.2f} s')
ax.set_xlabel('Time [s]')
ax.set_ylabel('SNR')
ax.set_title('SNR vs Re-weighted SNR — PyCBC')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Peak SNR          : {snr_common[mask_plot].max():.2f}")
print(f"Peak re-weighted  : {newsnr[mask_plot].max():.2f}")

## 12. Comparison: PyCBC vs ripple

Both notebooks now work on the same HDF5 data from the SparseBank Step-1 pipeline.

In [ ]:
summary = {
    'Data source'          : ('HDF5 from data_generation (Step 1)', 'HDF5 from data_generation (Step 1)'),
    'Waveform library'     : ('ripplegw (TaylorF2)', 'PyCBC (TaylorF2)'),
    'Template whitening'   : ('Manual / aLIGO PSD array', 'FrequencySeries / aLIGO PSD'),
    'Matched filter PSD'   : ('Flat (pre-whitened data)', 'Flat (pre-whitened data)'),
    'Matched filter impl.' : ('Manual IFFT', 'pycbc.filter.matched_filter'),
    'Match function'       : ('Manual IFFT', 'pycbc.filter.match'),
    'Chi-sq veto'          : ('Not implemented', 'pycbc.vetoes.power_chisq'),
    'Differentiable SNR'   : ('Yes — jax.grad', 'No'),
    'GPU acceleration'     : ('JAX (XLA)', 'CPU (FFTW)'),
}

print(f"{'Feature':<22}  {'ripple':<38}  {'PyCBC'}")
print('-' * 92)
for k, (v1, v2) in summary.items():
    print(f"{k:<22}  {v1:<38}  {v2}")

## Summary

### Data pipeline integration

| Step | Detail |
|------|--------|
| Input | `injected_data` — whitened H1+L1 strain, 55 s @ 512 Hz |
| True parameters | `mass_1`, `mass_2`, `chi1`, `chi2`, `distance`, `snr` |
| Template | TaylorF2, whitened with nominal aLIGO PSD |
| Matched filter PSD | Flat (data pre-whitened in Step 1) |
| Zero-padding | 55 s → 128 s so SNR peak near $t_c \approx 64$ s is reachable |

### Relation to SparseBank

SparseBank (Step 3) prunes the template bank by restricting it to templates whose chirp mass falls within the neural network's predicted range. The matched filter (Step 4) then runs the standard PyCBC pipeline on the reduced bank — exactly the workflow shown here, but with many fewer templates.